# KhazinaSmart — Data Loading & Cleaning

This notebook loads the Walmart Store Sales Forecasting dataset, performs initial inspection, merges dataframes, handles missing values, and saves a clean version for downstream analysis.

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


## 1. Load Raw Data

In [2]:
# If Kaggle data not available, generate synthetic data
if not os.path.exists('../data/raw/train.csv'):
    print('Kaggle data not found — generating synthetic data...')
    import subprocess
    subprocess.run(['python', '../generate_sample_data.py'], check=True)

train = pd.read_csv('../data/raw/train.csv')
stores = pd.read_csv('../data/raw/stores.csv')
features = pd.read_csv('../data/raw/features.csv')
print(f'train:    {train.shape}')
print(f'stores:   {stores.shape}')
print(f'features: {features.shape}')

train:    (29000, 5)
stores:   (10, 3)
features: (1450, 12)


## 2. Initial Inspection

In [3]:
for name, df in [('train', train), ('stores', stores), ('features', features)]:
    print(f'\n=== {name} ===')
    print(f'Shape: {df.shape}')
    print(f'Dtypes:\n{df.dtypes}')
    display(df.head(3))
    print(f'Nulls:\n{df.isnull().sum()}')


=== train ===
Shape: (29000, 5)
Dtypes:
Store             int64
Dept              int64
Date             object
Weekly_Sales    float64
IsHoliday          bool
dtype: object


,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,32055.67,False
1,1,1,2010-02-12,43299.71,True
2,1,1,2010-02-19,35970.89,False


Nulls:
Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64

=== stores ===
Shape: (10, 3)
Dtypes:
Store     int64
Type     object
Size      int64
dtype: object


,Store,Type,Size
0,1,A,150000
1,2,A,150000
2,3,A,150000


Nulls:
Store    0
Type     0
Size     0
dtype: int64

=== features ===
Shape: (1450, 12)
Dtypes:
Store             int64
Date             object
Temperature     float64
Fuel_Price      float64
MarkDown1       float64
MarkDown2       float64
MarkDown3       float64
MarkDown4       float64
MarkDown5       float64
CPI             float64
Unemployment    float64
IsHoliday          bool
dtype: object


,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,53.80,2.965,4956.83,0.0,2675.79,3924.64,4862.47,168.684,6.042,False
1,1,2010-02-12,51.76,3.263,0.00,0.0,0.00,0.00,992.09,162.300,11.917,True
2,1,2010-02-19,80.56,3.458,0.00,0.0,0.00,0.00,0.00,150.073,5.610,False


Nulls:
Store           0
Date            0
Temperature     0
Fuel_Price      0
MarkDown1       0
MarkDown2       0
MarkDown3       0
MarkDown4       0
MarkDown5       0
CPI             0
Unemployment    0
IsHoliday       0
dtype: int64


## 3. Merge DataFrames

In [4]:
train['Date'] = pd.to_datetime(train['Date'])
features['Date'] = pd.to_datetime(features['Date'])

df = train.merge(stores, on='Store', how='left')
df = df.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
print(f'Merged shape: {df.shape}')
display(df.head(3))

Merged shape: (29000, 16)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment
0,1,1,2010-02-05,32055.67,False,A,150000,53.80,2.965,4956.83,0.0,2675.79,3924.64,4862.47,168.684,6.042
1,1,1,2010-02-12,43299.71,True,A,150000,51.76,3.263,0.00,0.0,0.00,0.00,992.09,162.300,11.917
2,1,1,2010-02-19,35970.89,False,A,150000,80.56,3.458,0.00,0.0,0.00,0.00,0.00,150.073,5.610


## 4. Handle Missing Values

In [5]:
md_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
print('Nulls BEFORE fill:')
print(df[md_cols].isnull().sum())
for col in md_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)
print('\nNulls AFTER fill:')
print(df[md_cols].isnull().sum())

Nulls BEFORE fill:
MarkDown1    0
MarkDown2    0
MarkDown3    0
MarkDown4    0
MarkDown5    0
dtype: int64

Nulls AFTER fill:
MarkDown1    0
MarkDown2    0
MarkDown3    0
MarkDown4    0
MarkDown5    0
dtype: int64


## 5. Data Quality Report

In [6]:
print('=== KhazinaSmart Data Quality Report ===')
print(f'Total rows:        {len(df):,}')
print(f'Date range:        {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'Number of stores:  {df["Store"].nunique()}')
print(f'Number of depts:   {df["Dept"].nunique()}')
print(f'Holiday weeks (%): {df["IsHoliday"].mean()*100:.1f}%')
print(f'Weekly_Sales stats:')
print(f'  min:  {df["Weekly_Sales"].min():,.2f}')
print(f'  max:  {df["Weekly_Sales"].max():,.2f}')
print(f'  mean: {df["Weekly_Sales"].mean():,.2f}')

=== KhazinaSmart Data Quality Report ===
Total rows:        29,000
Date range:        2010-02-05 → 2012-11-09
Number of stores:  10
Number of depts:   20
Holiday weeks (%): 6.9%
Weekly_Sales stats:
  min:  2,172.74
  max:  128,094.06
  mean: 31,568.47


## 6. Save Clean Data

In [7]:
os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/walmart_clean.csv', index=False)
print(f'Saved: {df.shape[0]:,} rows, {df.shape[1]} columns → data/processed/walmart_clean.csv')

Saved: 29,000 rows, 16 columns → data/processed/walmart_clean.csv
